## Model Pipeline Training
### Baseline Model vs Resnet

ResNet with fine-tuning is outperforming the baseline (~85% vs ~75% val accuracy), less overfitting due to pretrained features + dropout, transfer learning seems to be working well for this small batched datasets.

In [1]:
# External Imports
import sys
from pathlib import Path
import glob
import torch
import torch.optim as optim
import torch.nn as nn
from torchinfo import summary

# Internal Imports
sys.path.insert(0, '../src')
from src.DataIntegrity import data_integrity_check
from src.Dataset import split_patients, get_dataloaders, save_cache
from src.Model import BaselineModel, build_resnet, fit, plot_loss_acc, save_weights

In [2]:
data_path = Path("../data/raw/lgg-mri-segmentation/kaggle_3m")
rejected_path = Path("../data/processed/data-integrity")

In [3]:
accepted_data, rejected_data = data_integrity_check(data_path)

[INFO] Data Integrity Checks: 100%|██████████| 112/112 [00:03<00:00, 34.30it/s]

[Info] Data Integrity Pipeline finished
[Info] Segments Accepted = 3929 | Segments Rejected = 0


In [5]:
save_cache(accepted_data, Path("../data/processed/cache/accepted_data.json"), Path.cwd().parent)
save_cache(rejected_data, Path("../data/processed/cache/rejected_data.json"), Path.cwd().parent)

[INFO] Cache saved to ..\data\processed\cache\accepted_data.json
[INFO] Cache saved to ..\data\processed\cache\rejected_data.json


In [6]:
SPLIT_SEED = 42
train_patients, val_patients, test_patients = split_patients(accepted_data, seed=SPLIT_SEED)

[INFO]  Splitting dataset with seed 42...
[INFO]  Train Set: 76  | Tumor Ratio: 0.352
[INFO]  Valid Set: 17  | Tumor Ratio: 0.360
[INFO]  Test Set:  17  | Tumor Ratio: 0.372


In [ ]:
BATCH_SIZE = 32
train_dataloader, val_dataloader, test_dataloader = get_dataloaders(train_patients, val_patients, test_patients, batch_size=BATCH_SIZE)

In [ ]:
baseline_model = BaselineModel()
device = "cuda" if torch.cuda.is_available() else "cpu"
baseline_model.to(device)

In [ ]:
baseline_epoch_data = fit(
    model = baseline_model,
    train_data = train_dataloader,
    valid_data = val_dataloader,
    optimizer = optim.Adam(baseline_model.parameters(), lr=1e-3),
    loss_fn = nn.CrossEntropyLoss(),
    device = device,
    epochs=15,
    patience=5
)

In [ ]:
resnet_model = build_resnet(num_classes=2)
resnet_model.to(device)

In [ ]:
resnet_epoch_data = fit(
    model = resnet_model,
    train_data = train_dataloader,
    valid_data = val_dataloader,
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=1e-3),
    loss_fn = nn.CrossEntropyLoss(),
    device = device,
    epochs=15,
    patience=5
)

In [ ]:
# Unfreeze Resnet layers and continue training
# Freeze everything
for param in resnet_model.parameters():
  param.requires_grad = False

# Replace fc (automatically unfrozen)
resnet_model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features=512, out_features=2)
)
# Unfreeze layer4
for param in resnet_model.layer4.parameters():
  param.requires_grad = True

resnet_model.to(device)

In [ ]:
resnet_epoch_data = fit(
    model = resnet_model,
    train_data = train_dataloader,
    valid_data = val_dataloader,
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=1e-4),
    loss_fn = nn.CrossEntropyLoss(),
    device = device,
    epochs=15,
    patience=5
)

In [ ]:
plot_loss_acc(baseline_epoch_data, "Baseline")
plot_loss_acc(resnet_epoch_data, "Resnet")

In [ ]:
save_weights("baseline", baseline_model, Path("../models"))
save_weights("resnet18", resnet_model, Path("../models"))